# 01 — Source Querying
Queries OSM via the Overpass API for infrastructure asset geometries across four sectors: power, water, telecom, and transportation.

**Output:** a CSV file in `../data/` that feeds directly into `02_imagery.ipynb`.

> ⚠️ **Before running:** set `OUTPUT_CSV` and `QUERY_BBOX` in the cell below. Change the filename each time you query a new region to avoid overwriting earlier data.

In [1]:
import sys
sys.path.insert(0, '..')   # so we can import from infra_curation/

import os
import pandas as pd
from sources import InfrastructureSource

## Configuration
**Change `OUTPUT_CSV` before each new query run.**
Convention: `../data/<region>_<scope>_assets.csv`

In [2]:
# !! CHANGE THIS before re-running for a different region/scope
# to avoid overwriting earlier query results.
#
# Examples:
#   '../data/maine_all_assets.csv'
#   '../data/portland_me_assets.csv'
#   '../data/new_hampshire_assets.csv'
#   '../data/maine_power_only.csv'

OUTPUT_CSV = '../data/maine_all_assets.csv'

# Bounding box for the query area: (min_lon, min_lat, max_lon, max_lat)
QUERY_BBOX = (-71.1, 43.0, -66.9, 47.5)   # Maine

os.makedirs('../data', exist_ok=True)
print(f'Output will be saved to: {OUTPUT_CSV}')

Output will be saved to: ../data/maine_all_assets.csv


## Query OSM

In [3]:
src = InfrastructureSource(bbox=QUERY_BBOX)
df = src.query_all()

  [power_substation] querying...
    HTTP error on https://overpass-api.de/api/interpreter: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter — trying next
  [power_substation] found 531 assets
  [water_treatment] querying...
  [water_treatment] found 202 assets
  [telecom_tower] querying...
    HTTP error on https://overpass-api.de/api/interpreter: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter — trying next
    timeout on https://overpass.kumi.systems/api/interpreter (attempt 1) — trying next
    connection error on https://maps.mail.ru/osm/tools/overpass/api/interpreter (attempt 1) — trying next
    all endpoints failed, waiting 5s before retry 2...
  [telecom_tower] found 1210 assets
  [transport_airport] querying...
    HTTP error on https://overpass-api.de/api/interpreter: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter — trying next
  [transport_airport] unexpected error: Expe

In [4]:
# Re-run a single sector that failed or was missing from the main query.
# Then append it to df and re-save.

RETRY_SECTOR = "transport_airport"   # change to whichever sector needs re-running

df_retry = src.query_sector(RETRY_SECTOR)

if not df_retry.empty:
    # drop any existing rows for this sector (avoids duplicates if re-running)
    df = df[df["asset_type"] != RETRY_SECTOR].copy()
    df = pd.concat([df, df_retry], ignore_index=True)
    
    # re-save
    df.drop(columns=["osm_tags"], errors="ignore").to_csv(OUTPUT_CSV, index=False)
    print(f"Added {len(df_retry)} {RETRY_SECTOR} assets. Total now: {len(df)}")
    print(df["asset_type"].value_counts().to_string())
else:
    print(f"Still no results for {RETRY_SECTOR} — try again later.")

  [transport_airport] querying...
    HTTP error on https://overpass-api.de/api/interpreter: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter — trying next
  [transport_airport] found 234 assets
Added 234 transport_airport assets. Total now: 2202
asset_type
telecom_tower              1210
power_substation            531
transport_airport           234
water_treatment             202
transport_train_station      25


## Inspect results

In [5]:
print(f'Total assets: {len(df)}')
print()
print('Asset counts by sector:')
print(df['asset_type'].value_counts().to_string())

Total assets: 2202

Asset counts by sector:
asset_type
telecom_tower              1210
power_substation            531
transport_airport           234
water_treatment             202
transport_train_station      25


In [6]:
# preview sample rows per sector
df.groupby('asset_type').head(3)[['asset_type','lat','lon','name']].reset_index(drop=True)

,asset_type,lat,lon,name
0,power_substation,46.999285,-67.609862,
1,power_substation,45.655859,-67.132922,
2,power_substation,45.891550,-67.457683,
3,water_treatment,46.149955,-67.574968,
4,water_treatment,44.546276,-69.693033,Maine Water Works Supply
5,water_treatment,44.774505,-69.710194,
6,telecom_tower,43.156617,-70.939677,WUNH-FM (Durham)
7,telecom_tower,43.053119,-70.767270,WHEB-AM (Portsmouth)
8,telecom_tower,43.027155,-70.880061,WERZ-FM (Exeter)
9,transport_train_station,47.361149,-70.025302,La Pocatière


## Save to CSV
The CSV is the handoff to `02_imagery.ipynb` — no need to re-query next time.

In [7]:
df.drop(columns=['osm_tags'], errors='ignore').to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df)} assets to {OUTPUT_CSV}')

Saved 2202 assets to ../data/maine_all_assets.csv
